# Sahit Bolla and Sahil Chute

#### Enchancing Scene Text Recognition Accuracy through Noise-Resilient Architectures

Imports

In [1]:
from pathlib import Path
import re
import numpy as np
import math

Setup

In [3]:
np.random.seed(42)

def build_image_details(image_directory: str, annotation_directory: str):
    image_details = []

    for image_path in sorted(Path(image_directory).glob("*.jpg")):
        annotation_path = Path(annotation_directory) / ("poly_gt_" + image_path.stem + ".txt")
        image_details.append((str(image_path), str(annotation_path)))
    
    return image_details

train_details_full = build_image_details("Total-Text/Train", "Total-Text/Annotation/groundtruth_polygonal_annotation/Train")
test_details = build_image_details("Total-Text/Test", "Total-Text/Annotation/groundtruth_polygonal_annotation/Test")

print("Train records:", len(train_details_full))
print("Test records:", len(test_details))

def split_train_val(image_details, val_ratio=0.12):
    clone_details = image_details.copy()
    np.random.shuffle(clone_details)

    val_size = math.floor(len(clone_details) * val_ratio)
    return clone_details[val_size:], clone_details[:val_size]

train_details, val_details = split_train_val(train_details_full)
print("Train records after split:", len(train_details))
print("Validation records after split:", len(val_details))

def parse_annotation(annotation_path: Path):
    entries = []

    for line in annotation_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue

        x_text = re.search(r"x:\s*\[\[(.*?)\]\]", line)
        y_text = re.search(r"y:\s*\[\[(.*?)\]\]", line)
        ornt_text = re.search(r"ornt:\s*\[u?'([^']*)'\]", line)
        transcription_text = re.search(r"transcriptions:\s*\[u?'(.*)'\]\s*$", line)

        if not (x_text and y_text and transcription_text and ornt_text):
            continue

        x_coords = [int(coord) for coord in x_text.group(1).split()]
        y_coords = [int(coord) for coord in y_text.group(1).split()]
        ornt = ornt_text.group(1)
        transcription = transcription_text.group(1)

        if (len(x_coords) != len(y_coords)
            or len(x_coords) < 3
            or len(y_coords) < 3
            or ornt.strip() == "#"
            or transcription.strip() == ""
            or transcription.strip() == "#"):
            continue

        polygon = np.stack([x_coords, y_coords], axis=1)
        entries.append((transcription, polygon))

    return entries
    
train_annotations = {Path(image_path).stem: parse_annotation(Path(annotation_path)) for image_path, annotation_path in train_details}
val_annotations = {Path(image_path).stem: parse_annotation(Path(annotation_path)) for image_path, annotation_path in val_details}
test_annotations = {Path(image_path).stem: parse_annotation(Path(annotation_path)) for image_path, annotation_path in test_details}

total_train_polygons = sum(len(ann) for ann in train_annotations.values())
total_val_polygons = sum(len(ann) for ann in val_annotations.values())
total_test_polygons = sum(len(ann) for ann in test_annotations.values())

print("Total polygons in train set:", total_train_polygons)
print("Total polygons in validation set:", total_val_polygons)
print("Total polygons in test set:", total_test_polygons)


Train records: 1255
Test records: 300
Train records after split: 1105
Validation records after split: 150
Total polygons in train set: 8279
Total polygons in validation set: 998
Total polygons in test set: 2175
